## Example 1: serverless_wsgi functionality test 1

In [ ]:
# Example 1: Basic WSGI application setup for serverless deployment
import serverless_wsgi
from io import StringIO
import sys

# Create a simple WSGI application
def simple_wsgi_app(environ, start_response):
    """Simple WSGI application for testing"""
    status = '200 OK'
    headers = [('Content-Type', 'text/html')]
    start_response(status, headers)
    
    # Get request information
    method = environ.get('REQUEST_METHOD', 'GET')
    path = environ.get('PATH_INFO', '/')
    query = environ.get('QUERY_STRING', '')
    
    response_body = f"""
    <html>
    <body>
        <h1>Simple WSGI Application</h1>
        <p>Method: {method}</p>
        <p>Path: {path}</p>
        <p>Query: {query}</p>
        <p>Server: {environ.get('SERVER_NAME', 'localhost')}</p>
    </body>
    </html>
    """
    return [response_body.encode('utf-8')]

print("Simple WSGI application created")

# Test basic module functionality
print(f"serverless_wsgi version info: {serverless_wsgi.__version__ if hasattr(serverless_wsgi, '__version__') else 'Unknown'}")

# Explore available functions
available_functions = [attr for attr in dir(serverless_wsgi) if not attr.startswith('_')]
print(f"Available functions: {available_functions}")

# Test with a mock event (typical AWS Lambda event structure)
mock_event = {
    'httpMethod': 'GET',
    'path': '/test',
    'queryStringParameters': {'param1': 'value1'},
    'headers': {
        'Content-Type': 'application/json',
        'User-Agent': 'test-agent'
    },
    'body': None,
    'isBase64Encoded': False
}

mock_context = type('MockContext', (), {
    'aws_request_id': 'test-request-id',
    'function_name': 'test-function',
    'memory_limit_in_mb': '128',
    'remaining_time_in_millis': lambda: 30000
})()

print(f"Mock event created: {mock_event['httpMethod']} {mock_event['path']}")
print(f"Mock context: {mock_context.function_name}")

# Test the handler function (main serverless_wsgi functionality).
# handle_request(app, event, context) is NOT a handler factory: it runs the WSGI app for ONE Lambda event
# and returns the Lambda proxy response dict. No AWS environment is needed for that.
response = serverless_wsgi.handle_request(simple_wsgi_app, mock_event, mock_context)
print(f"Handler response status: {response.get('statusCode', 'Unknown')}")
print(f"Handler response headers: {list(response.get('headers', {}).keys())}")
assert response['statusCode'] == 200, response
assert response['headers']['Content-Type'] == 'text/html', response['headers']
assert response['isBase64Encoded'] is False, response
for fragment in ('<p>Method: GET</p>', '<p>Path: /test</p>', '<p>Query: param1=value1</p>'):
    assert fragment in response['body'], (fragment, response['body'])

print("Example 1 completed successfully")

## Example 2: serverless_wsgi functionality test 2

In [ ]:
# Example 2: Flask application adaptation for serverless deployment
import serverless_wsgi

# Create a Flask-like application using raw WSGI
def flask_style_app(environ, start_response):
    """Flask-style WSGI application"""
    method = environ.get('REQUEST_METHOD', 'GET')
    path = environ.get('PATH_INFO', '/')
    query = environ.get('QUERY_STRING', '')
    
    # Route handling
    if path == '/':
        status = '200 OK'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        response = '{"message": "Welcome to Flask-style serverless app", "status": "success"}'
        
    elif path == '/api/health':
        status = '200 OK'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        response = '{"health": "ok", "timestamp": "2025-01-01T00:00:00Z"}'
        
    elif path.startswith('/api/echo'):
        status = '200 OK'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        
        # Parse query parameters manually
        params = {}
        if query:
            for pair in query.split('&'):
                if '=' in pair:
                    key, value = pair.split('=', 1)
                    params[key] = value
        
        response = f'{{"method": "{method}", "path": "{path}", "query_params": {str(params).replace("\'", "\"")}"}}'
        
    elif path == '/api/error':
        status = '500 Internal Server Error'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        response = '{"error": "Intentional server error", "code": 500}'
        
    else:
        status = '404 Not Found'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        response = f'{{"error": "Resource not found", "path": "{path}"}}'
    
    return [response.encode('utf-8')]

print("Flask-style WSGI application created")

# Test different routes with mock events
expected_status = {'/': 200, '/api/health': 200, '/api/echo': 200, '/api/error': 500, '/nonexistent': 404}
test_routes = [
    {'path': '/', 'method': 'GET'},
    {'path': '/api/health', 'method': 'GET'},
    {'path': '/api/echo', 'method': 'POST', 'query': 'name=test&id=123'},
    {'path': '/api/error', 'method': 'GET'},
    {'path': '/nonexistent', 'method': 'GET'}
]

print(f"\nTesting {len(test_routes)} different routes:")

for i, route in enumerate(test_routes):
    mock_event = {
        'httpMethod': route['method'],
        'path': route['path'],
        'queryStringParameters': None,
        'headers': {'Content-Type': 'application/json'},
        'body': None,
        'isBase64Encoded': False
    }
    
    # Add query parameters if specified
    if 'query' in route:
        mock_event['queryStringParameters'] = {}
        for param in route['query'].split('&'):
            if '=' in param:
                key, value = param.split('=', 1)
                mock_event['queryStringParameters'][key] = value

    print(f"\nRoute {i+1}: {route['method']} {route['path']}")
    
    # serverless_wsgi.handle_request(app, event, context) runs the WSGI app for this one Lambda event
    mock_context = type('Context', (), {'aws_request_id': f'req-{i+1}'})()
    response = serverless_wsgi.handle_request(flask_style_app, mock_event, mock_context)
    
    print(f"  Status: {response.get('statusCode', 'Unknown')}")
    print(f"  Headers: {response.get('headers', {})}")
    if response['statusCode'] < 400 and len(response['body']) < 200:
        print(f"  Body: {response['body']}")
    else:
        print(f"  Body length: {len(response.get('body', ''))}")
    assert response['statusCode'] == expected_status[route['path']], (route, response)
    assert response['headers']['Content-Type'] == 'application/json', response['headers']
    if route['path'] == '/api/echo':
        assert '"method": "POST"' in response['body'] and '"name": "test"' in response['body'], response['body']

# Test request/response transformation
print(f"\nTesting serverless_wsgi request transformation:")
sample_lambda_event = {
    'httpMethod': 'POST',
    'path': '/api/data',
    'queryStringParameters': {'format': 'json'},
    'headers': {
        'Content-Type': 'application/json',
        'Authorization': 'Bearer token123'
    },
    'body': '{"data": "test"}',
    'isBase64Encoded': False
}

print(f"Sample Lambda event structure:")
for key, value in sample_lambda_event.items():
    if isinstance(value, str) and len(value) > 50:
        print(f"  {key}: {value[:50]}...")
    else:
        print(f"  {key}: {value}")

print("Example 2 completed successfully")

## Example 3: serverless_wsgi functionality test 3

In [ ]:
# Example 3: Advanced request handling and middleware integration
import serverless_wsgi
import json
import base64
from urllib.parse import parse_qs

# Create middleware for request/response processing
class LoggingMiddleware:
    def __init__(self, app):
        self.app = app
        self.request_count = 0
    
    def __call__(self, environ, start_response):
        self.request_count += 1
        method = environ.get('REQUEST_METHOD', 'GET')
        path = environ.get('PATH_INFO', '/')
        
        print(f"Request #{self.request_count}: {method} {path}")
        
        # Call the wrapped application
        def custom_start_response(status, headers):
            print(f"Response #{self.request_count}: {status}")
            return start_response(status, headers)
        
        return self.app(environ, custom_start_response)

class AuthMiddleware:
    def __init__(self, app):
        self.app = app
    
    def __call__(self, environ, start_response):
        # Check for authorization header
        auth_header = None
        if 'HTTP_AUTHORIZATION' in environ:
            auth_header = environ['HTTP_AUTHORIZATION']
        
        # Simple auth check
        if environ.get('PATH_INFO', '').startswith('/api/secure'):
            if not auth_header or not auth_header.startswith('Bearer '):
                status = '401 Unauthorized'
                headers = [('Content-Type', 'application/json')]
                start_response(status, headers)
                return [b'{"error": "Authentication required"}']
        
        return self.app(environ, start_response)

# Main application with complex routing
def advanced_app(environ, start_response):
    method = environ.get('REQUEST_METHOD', 'GET')
    path = environ.get('PATH_INFO', '/')
    query_string = environ.get('QUERY_STRING', '')
    
    # Parse request body if present
    content_length = int(environ.get('CONTENT_LENGTH', 0))
    request_body = ''
    if content_length > 0 and environ.get('wsgi.input'):
        try:
            request_body = environ['wsgi.input'].read(content_length).decode('utf-8')
        except:
            request_body = ''
    
    # Parse query parameters
    query_params = parse_qs(query_string)
    
    # Route handling
    if path == '/api/upload':
        if method == 'POST':
            status = '200 OK'
            headers = [('Content-Type', 'application/json')]
            start_response(status, headers)
            
            # Simulate file upload handling
            response_data = {
                'message': 'File upload received',
                'body_length': len(request_body),
                'content_type': environ.get('CONTENT_TYPE', 'unknown')
            }
            return [json.dumps(response_data).encode('utf-8')]
        else:
            status = '405 Method Not Allowed'
            headers = [('Content-Type', 'application/json')]
            start_response(status, headers)
            return [b'{"error": "POST method required"}']
    
    elif path == '/api/search':
        status = '200 OK'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        
        # Handle search parameters
        query_term = query_params.get('q', [''])[0]
        page = int(query_params.get('page', ['1'])[0])
        limit = int(query_params.get('limit', ['10'])[0])
        
        # Simulate search results
        results = []
        if query_term:
            for i in range(limit):
                results.append({
                    'id': f'result_{page}_{i+1}',
                    'title': f'Search result {i+1} for "{query_term}"',
                    'score': round(1.0 - (i * 0.1), 2)
                })
        
        response_data = {
            'query': query_term,
            'page': page,
            'limit': limit,
            'results': results,
            'total': len(results)
        }
        
        return [json.dumps(response_data).encode('utf-8')]
    
    elif path == '/api/secure/profile':
        status = '200 OK'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        
        # This route is protected by AuthMiddleware
        auth_header = environ.get('HTTP_AUTHORIZATION', '')
        token = auth_header.replace('Bearer ', '') if auth_header.startswith('Bearer ') else ''
        
        response_data = {
            'user_id': 'user_123',
            'profile': 'User Profile Data',
            'token_used': token[:20] + '...' if len(token) > 20 else token,
            'timestamp': '2025-01-01T00:00:00Z'
        }
        
        return [json.dumps(response_data).encode('utf-8')]
    
    elif path == '/api/binary':
        status = '200 OK'
        headers = [('Content-Type', 'application/octet-stream')]
        start_response(status, headers)
        
        # Return binary data
        binary_data = bytes([i % 256 for i in range(100)])  # Sample binary data
        return [binary_data]
    
    else:
        status = '404 Not Found'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        return [json.dumps({'error': 'Not found', 'path': path}).encode('utf-8')]

# Wrap application with middleware
wrapped_app = LoggingMiddleware(AuthMiddleware(advanced_app))

print("Advanced WSGI application with middleware created")

# Test various scenarios
test_scenarios = [
    {
        'name': 'Search with parameters',
        'event': {
            'httpMethod': 'GET',
            'path': '/api/search',
            'queryStringParameters': {'q': 'python', 'page': '2', 'limit': '5'},
            'headers': {}  # an API Gateway proxy event always carries `headers`; serverless_wsgi requires it
        }
    },
    {
        'name': 'File upload simulation',
        'event': {
            'httpMethod': 'POST', 
            'path': '/api/upload',
            'headers': {'Content-Type': 'multipart/form-data'},
            'body': 'fake-file-content-data'
        }
    },
    {
        'name': 'Unauthorized secure access',
        'event': {
            'httpMethod': 'GET',
            'path': '/api/secure/profile',
            'headers': {}
        }
    },
    {
        'name': 'Authorized secure access',
        'event': {
            'httpMethod': 'GET',
            'path': '/api/secure/profile',
            'headers': {'Authorization': 'Bearer secret-token-12345'}
        }
    },
    {
        'name': 'Binary data request',
        'event': {
            'httpMethod': 'GET',
            'path': '/api/binary',
            'headers': {}
        }
    }
]

print(f"\nTesting {len(test_scenarios)} advanced scenarios:")
expected_status = {
    'Search with parameters': 200,
    'File upload simulation': 200,
    'Unauthorized secure access': 401,
    'Authorized secure access': 200,
    'Binary data request': 200,
}

for i, scenario in enumerate(test_scenarios):
    print(f"\nScenario {i+1}: {scenario['name']}")
    event = scenario['event']
    
    # serverless_wsgi.handle_request(app, event, context) runs the WSGI app for this one Lambda event
    mock_context = type('Context', (), {'aws_request_id': f'adv-req-{i+1}'})()
    response = serverless_wsgi.handle_request(wrapped_app, event, mock_context)
    
    print(f"  Status: {response.get('statusCode')}")
    print(f"  Headers: {list(response.get('headers', {}).keys())}")
    
    body = response.get('body', '')
    if response.get('isBase64Encoded'):
        print(f"  Base64 encoded body length: {len(body)}")
    elif response['statusCode'] >= 400:
        print(f"  Response body length: {len(body)}")
    elif len(body) < 300:
        print(f"  Response body: {body}")
    else:
        print(f"  Response body length: {len(body)}")
    assert response['statusCode'] == expected_status[scenario['name']], (scenario['name'], response)
    
    if scenario['name'] == 'Search with parameters':
        data = json.loads(body)
        assert (data['query'], data['page'], data['limit'], data['total']) == ('python', 2, 5, 5), data
    elif scenario['name'] == 'File upload simulation':
        data = json.loads(body)
        assert data['body_length'] == len('fake-file-content-data') and data['content_type'] == 'multipart/form-data', data
    elif scenario['name'] == 'Unauthorized secure access':
        assert json.loads(body) == {'error': 'Authentication required'}, body
    elif scenario['name'] == 'Authorized secure access':
        assert json.loads(body)['token_used'] == 'secret-token-12345', body
    elif scenario['name'] == 'Binary data request':
        assert response['isBase64Encoded'] is True, response
        assert base64.b64decode(body) == bytes(range(100)), body

# Test base64 encoding for binary responses
print(f"\nTesting base64 encoding for binary responses:")
test_binary_data = b"Binary data: \x00\x01\x02\xFF"
encoded_data = base64.b64encode(test_binary_data).decode('utf-8')
decoded_data = base64.b64decode(encoded_data)

print(f"Original binary length: {len(test_binary_data)}")
print(f"Base64 encoded length: {len(encoded_data)}")
print(f"Decoded matches original: {decoded_data == test_binary_data}")

print("Example 3 completed successfully")

## Example 4: serverless_wsgi functionality test 4

In [ ]:
# Example 4: Production deployment patterns and configuration
import serverless_wsgi
import os
import json
from datetime import datetime

# Simulated environment configuration
class ServerlessConfig:
    def __init__(self):
        self.debug = os.environ.get('DEBUG', 'false').lower() == 'true'
        self.cors_origins = os.environ.get('CORS_ORIGINS', '*').split(',')
        self.api_version = os.environ.get('API_VERSION', 'v1')
        self.max_request_size = int(os.environ.get('MAX_REQUEST_SIZE', '10485760'))  # 10MB
        self.timeout = int(os.environ.get('LAMBDA_TIMEOUT', '30'))
        self.log_level = os.environ.get('LOG_LEVEL', 'INFO')

config = ServerlessConfig()

# Production-ready WSGI application with error handling
def production_app(environ, start_response):
    try:
        method = environ.get('REQUEST_METHOD', 'GET')
        path = environ.get('PATH_INFO', '/')
        
        # Add CORS headers for all responses
        cors_headers = [
            ('Access-Control-Allow-Origin', config.cors_origins[0]),
            ('Access-Control-Allow-Methods', 'GET, POST, PUT, DELETE, OPTIONS'),
            ('Access-Control-Allow-Headers', 'Content-Type, Authorization'),
        ]
        
        # Handle OPTIONS (preflight) requests
        if method == 'OPTIONS':
            status = '200 OK'
            headers = [('Content-Type', 'application/json')] + cors_headers
            start_response(status, headers)
            return [b'{"message": "CORS preflight OK"}']
        
        # API versioning
        if not path.startswith(f'/{config.api_version}/'):
            status = '404 Not Found'
            headers = [('Content-Type', 'application/json')] + cors_headers
            start_response(status, headers)
            return [json.dumps({
                'error': 'API version not supported',
                'supported_version': config.api_version,
                'requested_path': path
            }).encode('utf-8')]
        
        # Remove version prefix from path for routing
        api_path = path[len(f'/{config.api_version}'):]
        
        # Health check endpoint
        if api_path == '/health':
            status = '200 OK'
            headers = [('Content-Type', 'application/json')] + cors_headers
            start_response(status, headers)
            
            health_data = {
                'status': 'healthy',
                'timestamp': datetime.utcnow().isoformat() + 'Z',
                'version': config.api_version,
                'environment': {
                    'debug': config.debug,
                    'timeout': config.timeout,
                    'max_request_size': config.max_request_size
                }
            }
            return [json.dumps(health_data).encode('utf-8')]
        
        # Metrics endpoint
        elif api_path == '/metrics':
            status = '200 OK'
            headers = [('Content-Type', 'application/json')] + cors_headers
            start_response(status, headers)
            
            # Simulate metrics collection
            metrics = {
                'requests_processed': 1234,
                'average_response_time_ms': 150,
                'error_rate': 0.02,
                'memory_usage_mb': 128,
                'cold_starts': 5,
                'timestamp': datetime.utcnow().isoformat() + 'Z'
            }
            return [json.dumps(metrics).encode('utf-8')]
        
        # API endpoint with request validation
        elif api_path == '/api/process':
            # Check content length
            content_length = int(environ.get('CONTENT_LENGTH', 0))
            if content_length > config.max_request_size:
                status = '413 Payload Too Large'
                headers = [('Content-Type', 'application/json')] + cors_headers
                start_response(status, headers)
                return [json.dumps({
                    'error': 'Request too large',
                    'max_size_bytes': config.max_request_size,
                    'received_size_bytes': content_length
                }).encode('utf-8')]
            
            if method == 'POST':
                # Read and validate request body
                request_data = {}
                if content_length > 0:
                    try:
                        body = environ['wsgi.input'].read(content_length).decode('utf-8')
                        request_data = json.loads(body) if body else {}
                    except (json.JSONDecodeError, UnicodeDecodeError) as e:
                        status = '400 Bad Request'
                        headers = [('Content-Type', 'application/json')] + cors_headers
                        start_response(status, headers)
                        return [json.dumps({
                            'error': 'Invalid JSON in request body',
                            'details': str(e)
                        }).encode('utf-8')]
                
                # Process the request
                status = '200 OK'
                headers = [('Content-Type', 'application/json')] + cors_headers
                start_response(status, headers)
                
                response_data = {
                    'status': 'processed',
                    'received_data': request_data,
                    'processed_at': datetime.utcnow().isoformat() + 'Z',
                    'processing_time_ms': 45
                }
                
                return [json.dumps(response_data).encode('utf-8')]
            else:
                status = '405 Method Not Allowed'
                headers = [('Content-Type', 'application/json')] + cors_headers
                start_response(status, headers)
                return [json.dumps({'error': 'Only POST method allowed'}).encode('utf-8')]
        
        # Configuration endpoint (debug mode only)
        elif api_path == '/config' and config.debug:
            status = '200 OK'
            headers = [('Content-Type', 'application/json')] + cors_headers
            start_response(status, headers)
            
            config_data = {
                'debug': config.debug,
                'api_version': config.api_version,
                'cors_origins': config.cors_origins,
                'max_request_size': config.max_request_size,
                'timeout': config.timeout,
                'log_level': config.log_level
            }
            return [json.dumps(config_data, indent=2).encode('utf-8')]
        
        else:
            status = '404 Not Found'
            headers = [('Content-Type', 'application/json')] + cors_headers
            start_response(status, headers)
            return [json.dumps({
                'error': 'Endpoint not found',
                'path': api_path,
                'available_endpoints': ['/health', '/metrics', '/api/process']
            }).encode('utf-8')]
    
    except Exception as e:
        # Global error handler
        if config.debug:
            error_detail = str(e)
            import traceback
            stack_trace = traceback.format_exc()
        else:
            error_detail = 'Internal server error'
            stack_trace = None
        
        status = '500 Internal Server Error'
        headers = [('Content-Type', 'application/json')] + cors_headers
        start_response(status, headers)
        
        error_response = {
            'error': error_detail,
            'timestamp': datetime.utcnow().isoformat() + 'Z'
        }
        
        if stack_trace and config.debug:
            error_response['stack_trace'] = stack_trace
        
        return [json.dumps(error_response).encode('utf-8')]

print(f"Production WSGI application created")
print(f"Configuration: Debug={config.debug}, API Version={config.api_version}, CORS Origins={config.cors_origins}")

# Test production scenarios
production_scenarios = [
    {
        'name': 'Health check',
        'event': {
            'httpMethod': 'GET',
            'path': f'/{config.api_version}/health',
            'headers': {}
        }
    },
    {
        'name': 'CORS preflight',
        'event': {
            'httpMethod': 'OPTIONS',
            'path': f'/{config.api_version}/api/process',
            'headers': {'Origin': 'https://example.com'}
        }
    },
    {
        'name': 'Valid API request',
        'event': {
            'httpMethod': 'POST',
            'path': f'/{config.api_version}/api/process',
            'headers': {'Content-Type': 'application/json'},
            'body': json.dumps({'action': 'test', 'data': [1, 2, 3]})
        }
    },
    {
        'name': 'Invalid JSON request',
        'event': {
            'httpMethod': 'POST',
            'path': f'/{config.api_version}/api/process',
            'headers': {'Content-Type': 'application/json'},
            'body': 'invalid-json{'
        }
    },
    {
        'name': 'Unsupported API version',
        'event': {
            'httpMethod': 'GET',
            'path': '/v99/health',
            'headers': {}
        }
    },
    {
        'name': 'Metrics endpoint',
        'event': {
            'httpMethod': 'GET',
            'path': f'/{config.api_version}/metrics',
            'headers': {}
        }
    }
]

print(f"\nTesting {len(production_scenarios)} production scenarios:")

for i, scenario in enumerate(production_scenarios):
    print(f"\nScenario {i+1}: {scenario['name']}")
    event = scenario['event']
    
    try:
        # Manual WSGI testing (since we may not have actual serverless_wsgi handler)
        environ = {
            'REQUEST_METHOD': event['httpMethod'],
            'PATH_INFO': event['path'],
            'QUERY_STRING': '',
            'CONTENT_TYPE': event.get('headers', {}).get('Content-Type', ''),
            'CONTENT_LENGTH': str(len(event.get('body', ''))),
            'wsgi.version': (1, 0),
            'wsgi.input': None,
            'wsgi.errors': None,
            'wsgi.multithread': False,
            'wsgi.multiprocess': True,
            'wsgi.run_once': False
        }
        
        # Add request body
        if event.get('body'):
            from io import BytesIO
            environ['wsgi.input'] = BytesIO(event['body'].encode('utf-8'))
        
        # Add headers
        for header_name, header_value in event.get('headers', {}).items():
            env_name = f"HTTP_{header_name.upper().replace('-', '_')}"
            environ[env_name] = header_value
        
        status_list = []
        headers_list = []
        
        def start_response(status, headers):
            status_list.append(status)
            headers_list.append(headers)
        
        response_body = production_app(environ, start_response)
        
        if status_list:
            print(f"  Status: {status_list[0]}")
            
            # Show important headers
            headers_dict = dict(headers_list[0])
            important_headers = ['Content-Type', 'Access-Control-Allow-Origin']
            for header in important_headers:
                if header in headers_dict:
                    print(f"  {header}: {headers_dict[header]}")
            
            body = b''.join(response_body).decode('utf-8')
            if len(body) < 400:
                # Try to pretty print JSON
                try:
                    parsed = json.loads(body)
                    print(f"  Body: {json.dumps(parsed, indent=2)[:300]}{'...' if len(body) > 300 else ''}")
                except:
                    print(f"  Body: {body[:300]}{'...' if len(body) > 300 else ''}")
                    raise
            else:
                print(f"  Body length: {len(body)} characters")
    
    except Exception as e:
        print(f"  Error: {e}")
        raise

# Test Lambda function wrapper pattern
print(f"\nLambda deployment pattern example:")
print(f"Creating lambda_handler function for AWS deployment...")

# This is how you would typically set up the Lambda handler
def lambda_handler(event, context):
    """AWS Lambda handler function"""
    try:
        if hasattr(serverless_wsgi, 'handle_request'):
            # Using serverless_wsgi
            handler = serverless_wsgi.handle_request(production_app)
            return handler(event, context)
        else:
            # Fallback manual handling
            return {
                'statusCode': 200,
                'headers': {'Content-Type': 'application/json'},
                'body': json.dumps({
                    'message': 'Manual Lambda handler',
                    'event_keys': list(event.keys()) if event else [],
                    'context_type': str(type(context))
                })
            }
    except Exception as e:
        return {
            'statusCode': 500,
            'headers': {'Content-Type': 'application/json'},
            'body': json.dumps({'error': str(e)})
        }

print("Lambda handler function created: lambda_handler(event, context)")
print("Ready for deployment to AWS Lambda")

print("Example 4 completed successfully")

## Example 5: serverless_wsgi functionality test 5

In [ ]:
# Example 5: Performance optimization and monitoring integration
import serverless_wsgi
import time
import json
import gzip
from io import BytesIO

# Performance monitoring class
class PerformanceMonitor:
    def __init__(self):
        self.request_times = []
        self.total_requests = 0
        self.error_count = 0
        
    def record_request(self, duration, status_code):
        self.request_times.append(duration)
        self.total_requests += 1
        if status_code >= 400:
            self.error_count += 1
    
    def get_stats(self):
        if not self.request_times:
            return {'message': 'No requests recorded'}
            
        avg_time = sum(self.request_times) / len(self.request_times)
        max_time = max(self.request_times)
        min_time = min(self.request_times)
        error_rate = self.error_count / self.total_requests if self.total_requests > 0 else 0
        
        return {
            'total_requests': self.total_requests,
            'average_response_time_ms': round(avg_time * 1000, 2),
            'max_response_time_ms': round(max_time * 1000, 2),
            'min_response_time_ms': round(min_time * 1000, 2),
            'error_rate': round(error_rate, 4),
            'error_count': self.error_count
        }

# Global performance monitor
perf_monitor = PerformanceMonitor()

# Compression middleware
class CompressionMiddleware:
    def __init__(self, app, min_size=1024):
        self.app = app
        self.min_size = min_size
    
    def __call__(self, environ, start_response):
        # Check if client accepts gzip
        accept_encoding = environ.get('HTTP_ACCEPT_ENCODING', '')
        use_gzip = 'gzip' in accept_encoding.lower()
        
        # Capture response
        response_data = []
        status_code = [None]
        headers_list = [None]
        
        def capture_start_response(status, headers):
            status_code[0] = status
            headers_list[0] = headers
            return start_response(status, headers)
        
        # Get response from wrapped app
        app_response = self.app(environ, capture_start_response)
        
        # Collect response body
        body_parts = []
        for part in app_response:
            body_parts.append(part)
        
        body = b''.join(body_parts)
        
        # Apply compression if beneficial
        if use_gzip and len(body) >= self.min_size:
            # Check content type
            content_type = None
            for header_name, header_value in headers_list[0]:
                if header_name.lower() == 'content-type':
                    content_type = header_value.lower()
                    break
            
            # Only compress text-based content
            compressible_types = ['application/json', 'text/', 'application/javascript', 'application/xml']
            should_compress = any(ct in content_type for ct in compressible_types) if content_type else False
            
            if should_compress:
                # Compress the body
                compressed_buffer = BytesIO()
                with gzip.GzipFile(fileobj=compressed_buffer, mode='wb') as gzip_file:
                    gzip_file.write(body)
                
                compressed_body = compressed_buffer.getvalue()
                
                # Only use compressed version if it's actually smaller
                if len(compressed_body) < len(body):
                    # Update headers
                    new_headers = []
                    for header_name, header_value in headers_list[0]:
                        if header_name.lower() not in ['content-length', 'content-encoding']:
                            new_headers.append((header_name, header_value))
                    
                    new_headers.append(('Content-Encoding', 'gzip'))
                    new_headers.append(('Content-Length', str(len(compressed_body))))
                    
                    # Start new response with updated headers
                    start_response(status_code[0], new_headers)
                    return [compressed_body]
        
        return [body]

# High-performance WSGI application
def optimized_app(environ, start_response):
    start_time = time.time()
    method = environ.get('REQUEST_METHOD', 'GET')
    path = environ.get('PATH_INFO', '/')
    
    try:
        # Caching simulation
        if path == '/api/cached-data':
            # Check if client sent If-None-Match header BEFORE starting the response: WSGI allows one
            # start_response() per response, and the 200 used to be sent first, then a second 304 call.
            if_none_match = environ.get('HTTP_IF_NONE_MATCH')
            if if_none_match == '"data-v1.0"':
                status = '304 Not Modified'
                headers = [('Cache-Control', 'public, max-age=3600')]
                start_response(status, headers)
                return [b'']
            
            status = '200 OK'
            headers = [
                ('Content-Type', 'application/json'),
                ('Cache-Control', 'public, max-age=3600'),  # 1 hour cache
                ('ETag', '"data-v1.0"')
            ]
            start_response(status, headers)
            
            # Generate large response for compression testing
            data = {
                'items': [
                    {'id': i, 'name': f'Item {i}', 'description': f'This is a detailed description for item {i}' * 5}
                    for i in range(100)
                ],
                'metadata': {
                    'total': 100,
                    'page': 1,
                    'timestamp': time.time(),
                    'cache_info': 'This response is cached for 1 hour'
                }
            }
            
            return [json.dumps(data).encode('utf-8')]
        
        # Database simulation with connection pooling concept
        elif path == '/api/database':
            # Simulate database query time
            query_time = 0.05  # 50ms
            time.sleep(query_time)
            
            status = '200 OK'
            headers = [('Content-Type', 'application/json')]
            start_response(status, headers)
            
            # Simulate database results
            results = {
                'query_time_ms': query_time * 1000,
                'connection_pool': 'active',
                'results': [
                    {'user_id': i, 'username': f'user_{i}', 'active': True}
                    for i in range(1, 21)
                ],
                'performance_hints': [
                    'Query optimized with indexes',
                    'Connection reused from pool',
                    'Result set limited to 20 items'
                ]
            }
            
            return [json.dumps(results).encode('utf-8')]
        
        # Memory-intensive operation simulation
        elif path == '/api/intensive':
            status = '200 OK'
            headers = [('Content-Type', 'application/json')]
            start_response(status, headers)
            
            # Simulate CPU-intensive work
            result = sum(i * i for i in range(10000))
            
            # Simulate memory usage tracking
            import sys
            
            response_data = {
                'computation_result': result,
                'memory_info': {
                    'python_objects': len(gc.get_objects()) if 'gc' in globals() else 'N/A'
                },
                'performance_tips': [
                    'Use generators for large datasets',
                    'Implement result caching',
                    'Consider async processing for CPU-intensive tasks'
                ]
            }
            
            return [json.dumps(response_data).encode('utf-8')]
        
        # Performance monitoring endpoint
        elif path == '/api/performance':
            status = '200 OK'
            headers = [('Content-Type', 'application/json')]
            start_response(status, headers)
            
            stats = perf_monitor.get_stats()
            current_time = time.time() - start_time
            
            performance_data = {
                'current_request_time_ms': round(current_time * 1000, 2),
                'aggregate_stats': stats,
                'optimization_status': {
                    'compression_enabled': True,
                    'caching_enabled': True,
                    'connection_pooling': 'simulated',
                    'monitoring_active': True
                },
                'recommendations': [
                    'Enable CloudWatch monitoring',
                    'Set up X-Ray tracing',
                    'Configure auto-scaling based on metrics',
                    'Implement circuit breakers for external services'
                ]
            }
            
            return [json.dumps(performance_data, indent=2).encode('utf-8')]
        
        # Load testing endpoint
        elif path == '/api/load-test':
            status = '200 OK'
            headers = [('Content-Type', 'application/json')]
            start_response(status, headers)
            
            # Simulate variable response times
            import random
            processing_delay = random.uniform(0.01, 0.1)  # 10-100ms
            time.sleep(processing_delay)
            
            response_data = {
                'request_id': f"req_{int(time.time() * 1000)}",
                'processing_time_ms': round(processing_delay * 1000, 2),
                'server_load': random.uniform(0.1, 0.8),
                'memory_usage_mb': random.randint(50, 200),
                'status': 'processed'
            }
            
            return [json.dumps(response_data).encode('utf-8')]
        
        else:
            status = '404 Not Found'
            headers = [('Content-Type', 'application/json')]
            start_response(status, headers)
            return [json.dumps({'error': 'Endpoint not found'}).encode('utf-8')]
    
    except Exception as e:
        status = '500 Internal Server Error'
        headers = [('Content-Type', 'application/json')]
        start_response(status, headers)
        
        return [json.dumps({
            'error': str(e),
            'request_time_ms': round((time.time() - start_time) * 1000, 2)
        }).encode('utf-8')]
    
    finally:
        # Record performance metrics
        end_time = time.time()
        duration = end_time - start_time
        status_code = 200  # Default, would be extracted from actual status in real implementation
        perf_monitor.record_request(duration, status_code)

# Wrap application with compression middleware
compressed_app = CompressionMiddleware(optimized_app)

print("High-performance WSGI application with compression middleware created")

# Test performance scenarios
performance_scenarios = [
    {
        'name': 'Cached data (large response)',
        'event': {
            'httpMethod': 'GET',
            'path': '/api/cached-data',
            'headers': {'Accept-Encoding': 'gzip, deflate'}
        }
    },
    {
        'name': 'Cached data with ETag (304 response)',
        'event': {
            'httpMethod': 'GET', 
            'path': '/api/cached-data',
            'headers': {
                'Accept-Encoding': 'gzip, deflate',
                'If-None-Match': '"data-v1.0"'
            }
        }
    },
    {
        'name': 'Database query simulation',
        'event': {
            'httpMethod': 'GET',
            'path': '/api/database',
            'headers': {}
        }
    },
    {
        'name': 'CPU-intensive operation',
        'event': {
            'httpMethod': 'GET',
            'path': '/api/intensive',
            'headers': {}
        }
    },
    {
        'name': 'Performance monitoring',
        'event': {
            'httpMethod': 'GET',
            'path': '/api/performance',
            'headers': {}
        }
    }
]

print(f"\nTesting {len(performance_scenarios)} performance scenarios:")

import gc  # For garbage collection info

for i, scenario in enumerate(performance_scenarios):
    print(f"\nScenario {i+1}: {scenario['name']}")
    event = scenario['event']
    
    scenario_start = time.time()
    
    try:
        # Manual WSGI testing with performance tracking
        environ = {
            'REQUEST_METHOD': event['httpMethod'],
            'PATH_INFO': event['path'],
            'QUERY_STRING': '',
            'wsgi.version': (1, 0),
            'wsgi.input': BytesIO(),
            'wsgi.errors': None
        }
        
        # Add headers to environ
        for header_name, header_value in event.get('headers', {}).items():
            env_name = f"HTTP_{header_name.upper().replace('-', '_')}"
            environ[env_name] = header_value
        
        status_list = []
        headers_list = []
        
        def start_response(status, headers):
            status_list.append(status)
            headers_list.append(headers)
        
        response_body = compressed_app(environ, start_response)
        
        scenario_end = time.time()
        scenario_duration = scenario_end - scenario_start
        
        if status_list:
            print(f"  Status: {status_list[0]}")
            print(f"  Processing time: {round(scenario_duration * 1000, 2)}ms")
            
            # Check for compression headers
            headers_dict = dict(headers_list[0])
            if 'Content-Encoding' in headers_dict:
                print(f"  Compression: {headers_dict['Content-Encoding']}")
            
            if 'Cache-Control' in headers_dict:
                print(f"  Cache-Control: {headers_dict['Cache-Control']}")
            
            body = b''.join(response_body)
            print(f"  Response size: {len(body)} bytes")
            
            if len(body) < 500 and status_list[0].startswith('200') and 'Content-Encoding' not in headers_dict:
                # every 200 from optimized_app is an uncompressed JSON document
                parsed = json.loads(body.decode('utf-8'))
                if 'performance_tips' in parsed or 'recommendations' in parsed:
                    print("  Contains performance optimization guidance")
            if scenario['name'] == 'Cached data with ETag (304 response)':
                assert len(status_list) == 1 and status_list[0] == '304 Not Modified' and body == b'', (status_list, body)
    
    except Exception as e:
        print(f"  Error: {e}")
        raise  # this per-scenario catch-all used to swallow a failure in the checks above

# Final performance summary
print(f"\nFinal Performance Summary:")
final_stats = perf_monitor.get_stats()
for key, value in final_stats.items():
    print(f"  {key}: {value}")

print("\nPerformance Optimization Features Demonstrated:")
print("  ✓ Response compression (gzip)")
print("  ✓ HTTP caching (ETag, Cache-Control)")
print("  ✓ Performance monitoring")
print("  ✓ Request/response time tracking")
print("  ✓ Error rate monitoring")
print("  ✓ Memory usage awareness")
print("  ✓ Database connection simulation")
print("  ✓ CPU-intensive operation handling")

print("Example 5 completed successfully")